# [Baseline] Task 3 · 다이나믹 프라이싱을 통한 입장 수익 최대화 제안서 작성

**수요 배수 구조**
$$M_t = M_0 \cdot \exp\!\left(\sum_j \delta_j V_{j,t}\right)$$

**주의**: `gamma`/`u` 파라미터는 최적해가 가격 상·하한에 붙지 않도록(내부해가 나오도록) 재보정되었다.

In [46]:
# =========================================================
# 0. import
# =========================================================
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [47]:
# =========================================================
# 1. [출제자 고정 영역] 2024 시즌 잠재관중 M0 테이블
#    - 본 baseline은 2024 시즌 단일 시즌만 사용한다
#    - 2025 시즌은 Phase 1의 정답(hold-out) 구간이므로 별도 관리
# =========================================================
M0_df = pd.DataFrame({
    'team':   ['Panthers', 'Chiefs', 'Rams', 'Titans', 'Dolphins',
               'Jaguars', 'Falcons', 'Chargers', 'Bengals', 'Ravens'],
    'season': 2024,
    'M0':     [15012, 12163, 20471, 10105, 17883, 17575, 17441, 18550, 10544, 12370],
})

print('M0 테이블:', M0_df.shape, '= 10팀 x 1시즌 (2024)')
M0_df.head()

M0 테이블: (10, 3) = 10팀 x 1시즌 (2024)


,team,season,M0
0,Panthers,2024,15012
1,Chiefs,2024,12163
2,Rams,2024,20471
3,Titans,2024,10105
4,Dolphins,2024,17883


In [48]:
# =========================================================
# 2. [출제자 고정 영역] 2024 시즌 구장 수용인원 테이블
# =========================================================
CAPACITY_BASE = {
    'Panthers': 20500, 'Chiefs': 18700, 'Rams': 23750, 'Titans': 17900, 'Dolphins': 23000, 'Jaguars': 23750, 'Falcons': 22900, 'Chargers': 24000, 'Bengals': 16000, 'Ravens': 12000}

CAPACITY_df = pd.DataFrame(
    [{'team': tm, 'season': 2024, 'capacity': cap} for tm, cap in CAPACITY_BASE.items()]
)

print('Capacity 테이블:', CAPACITY_df.shape, '= 10팀 x 1시즌 (2024)')
CAPACITY_df.head()

Capacity 테이블: (10, 3) = 10팀 x 1시즌 (2024)


,team,season,capacity
0,Panthers,2024,20500
1,Chiefs,2024,18700
2,Rams,2024,23750
3,Titans,2024,17900
4,Dolphins,2024,23000


In [49]:
# ==============================================================================
# 3. [출제자 고정 영역] 글로벌 설정 및 구조 파라미터
# ==============================================================================
CONFIG = {
    'gamma_G': 0.00018,   # 일반석 가격 민감도
    'gamma_P': 0.00011,   # 프리미엄석 가격 민감도
    'u_G': 3.0,           # 일반석 기본 효용
    'u_P': 3.7,           # 프리미엄석 기본 효용
    'alpha_G': 0.3,       # 일반석 잔여재고 감쇠계수
    'alpha_P': 0.1,       # 프리미엄석 잔여재고 감쇠계수
    'prop_G': 0.85,       # 구장 내 일반석 비율
    'prop_P': 0.15,       # 구장 내 프리미엄석 비율
    'share_D7': 0.4,      # D-7 시점 시장 배분 비율
    'share_D3': 0.6,      # D-3 시점 시장 배분 비율
    'p_min_ratio': 0.7,   # 최소 가격 (기본가 대비)
    'p_max_ratio': 1.5,   # 최대 가격 (기본가 대비)
}

BASE_PRICES = {'일반석': 12000, '프리미엄석': 35000}

# log_shock(가상지표 합)의 상하한
SHOCK_CLIP = (-0.6, 0.8)

In [50]:
# ==============================================================================
# 4. [출제자 고정 영역] 방법론 데모용 예시 데이터셋 (보고서 4.2절 재현용)
#
#  주의: 이 데이터셋은 방법론을 보여주기 위한 예시일 뿐, 실제 Phase 3 문제 시나리오가 아니다.
#        실제 문제 시나리오는 df_games_scenarios 를 참고할 것.
# ==============================================================================
df_games_demo = pd.DataFrame([
    # 라이벌전 (Jaguars vs Rams)
    {'date': '2024-05-11', 'season': 2024, 'home_team': 'Jaguars', 'away_team': 'Rams', 'day_of_week': '토', 'home_rank': 2, 'away_rank': 3},
    # 평상시 대조군 (Jaguars vs Falcons)
    {'date': '2024-05-18', 'season': 2024, 'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토', 'home_rank': 2, 'away_rank': 5},
])
df_games_demo

,date,season,home_team,away_team,day_of_week,home_rank,away_rank
0,2024-05-11,2024,Jaguars,Rams,토,2,3
1,2024-05-18,2024,Jaguars,Falcons,토,2,5


In [51]:
# ==============================================================================
# 5. [출제자 고정 영역] 핵심 수요 모델 (Multinomial Logit)
# ==============================================================================
def multinomial_logit_demand(M_period, p_G, p_P, gamma_eff_G, gamma_eff_P, cfg):
    """
    출제 가이드라인에 정의된 다항 로짓(Multinomial Logit) 수요 모형 수식을 연산다.

    Parameters:
    - M_period: 해당 판매 기간(D-7 혹은 D-3)에 할당된 시장 수요 규모
    - p_G, p_P: 일반석 및 프리미엄석 가격
    - gamma_eff_G, gamma_eff_P: 희소성이 반영된 유효 가격 민감도

    Returns:
    - Q_G, Q_P: 일반석 및 프리미엄석의 수요 기댓값 (물리적 한도 적용 전)
    """
    v_G = cfg['u_G'] - gamma_eff_G * p_G
    v_P = cfg['u_P'] - gamma_eff_P * p_P

    denom = 1.0 + np.exp(v_G) + np.exp(v_P)

    Q_G = M_period * np.exp(v_G) / denom
    Q_P = M_period * np.exp(v_P) / denom
    return Q_G, Q_P

In [52]:
# ==============================================================================
# 6. [출제자 고정 영역] 기대수익 목적함수
# ==============================================================================
def simulate_sale(prices, M_t, cap_total, cfg=CONFIG):
    p_D7_G, p_D7_P, p_D3_G, p_D3_P = prices

    cap_G = cap_total * cfg['prop_G']
    cap_P = cap_total * cfg['prop_P']

    # ---------- [1단계] D-7 시점 ----------
    f_G_D7 = 1.0
    f_P_D7 = 1.0
    gamma_eff_G_D7 = cfg['gamma_G'] / f_G_D7
    gamma_eff_P_D7 = cfg['gamma_P'] / f_P_D7

    M_D7 = M_t * cfg['share_D7']
    Q_D7_G_pred, Q_D7_P_pred = multinomial_logit_demand(
        M_D7, p_D7_G, p_D7_P, gamma_eff_G_D7, gamma_eff_P_D7, cfg)

    Q_D7_G = min(Q_D7_G_pred, cap_G)
    Q_D7_P = min(Q_D7_P_pred, cap_P)

    stock_ratio_G = (cap_G - Q_D7_G) / cap_G if cap_G > 0 else 0.0
    stock_ratio_P = (cap_P - Q_D7_P) / cap_P if cap_P > 0 else 0.0

    f_G_D3 = 1.0 + cfg['alpha_G'] * (1.0 - stock_ratio_G)
    f_P_D3 = 1.0 + cfg['alpha_P'] * (1.0 - stock_ratio_P)

    gamma_eff_G_D3 = cfg['gamma_G'] / f_G_D3
    gamma_eff_P_D3 = cfg['gamma_P'] / f_P_D3

    M_D3 = M_t * cfg['share_D3']
    Q_D3_G_pred, Q_D3_P_pred = multinomial_logit_demand(
        M_D3, p_D3_G, p_D3_P, gamma_eff_G_D3, gamma_eff_P_D3, cfg)

    Q_D3_G = min(Q_D3_G_pred, cap_G - Q_D7_G)
    Q_D3_P = min(Q_D3_P_pred, cap_P - Q_D7_P)

    revenue = (p_D7_G * Q_D7_G + p_D7_P * Q_D7_P +
               p_D3_G * Q_D3_G + p_D3_P * Q_D3_P)

    return {
        # 수익
        'revenue': revenue,
        # 시장 배분
        'M_D7': M_D7, 'M_D3': M_D3,
        # 좌석 용량
        'cap_G': cap_G, 'cap_P': cap_P,
        # 시점별 판매량
        'Q_D7_G': Q_D7_G, 'Q_D7_P': Q_D7_P,
        'Q_D3_G': Q_D3_G, 'Q_D3_P': Q_D3_P,
        'total_audience': Q_D7_G + Q_D7_P + Q_D3_G + Q_D3_P,
        # 희소성 메커니즘 (D-7 판매 결과 -> D-3 민감도)
        'r_G': stock_ratio_G, 'r_P': stock_ratio_P,      # 잔여재고율
        'f_G': f_G_D3, 'f_P': f_P_D3,                    # 희소성 계수
        'gamma_eff_G_D3': gamma_eff_G_D3,                # 유효 가격민감도
        'gamma_eff_P_D3': gamma_eff_P_D3,
    }


def expected_revenue_objective(prices, M_t, cap_total, cfg):
    """
    최적화 엔진(minimize)에 넘기는 목적함수. 음의 총 기대수익을 반환한다.
    """
    return -simulate_sale(prices, M_t, cap_total, cfg)['revenue']


def revenue_detail(prices, M_t, cap_total, cfg=CONFIG):
    """
    진단/보고서용. simulate_sale 의 결과를 그대로 돌려준다.
    (최적화에는 관여하지 않으며, 목적함수와 완전히 동일한 계산 결과를 보장한다.)
    """
    return simulate_sale(prices, M_t, cap_total, cfg)

In [53]:
# ==============================================================================
# 7. [출제자 고정 영역] SLSQP 최적화 엔진
# ==============================================================================
def optimize_single_game(M_t, cap_total, cfg=CONFIG, multi_start=True):
    bG, bP = BASE_PRICES['일반석'], BASE_PRICES['프리미엄석']
    lo, hi = cfg['p_min_ratio'], cfg['p_max_ratio']

    bounds = [(bG * lo, bG * hi), (bP * lo, bP * hi),
              (bG * lo, bG * hi), (bP * lo, bP * hi)]

    constraints = [
        {'type': 'ineq', 'fun': lambda x: x[1] - x[0]},   # C1: p_D7_P >= p_D7_G
        {'type': 'ineq', 'fun': lambda x: x[3] - x[2]},   # C1: p_D3_P >= p_D3_G
    ]

    starts = [[bG, bP, bG, bP]]
    if multi_start:
        starts += [[bG * 1.2, bP * 1.2, bG * 1.3, bP * 1.3],
                   [bG * 0.8, bP * 0.8, bG * 0.9, bP * 0.9],
                   [bG * 0.5, bP * 0.5, bG * 0.4, bP * 0.4]]

    best = None
    for x0 in starts:
        res = minimize(expected_revenue_objective, x0,
                       args=(M_t, cap_total, cfg),
                       method='SLSQP', bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        if res.success and (best is None or res.fun < best.fun):
            best = res
    return best if best is not None else res

In [54]:
# ==============================================================================
# 8. [출제자 고정 영역] 시뮬레이션용 통합 실행 파이프라인
# ==============================================================================
RIVALRY_PAIRS = [
    ('Jaguars', 'Rams'),   # 전통의 라이벌 매치업 (연고지 라이벌)
]

def my_virtual_indicators(row, cfg=CONFIG):
    log_shock = 0.0

    # ---- [지표 1] 라이벌 매치업 효과 (선별적용) ----
    # delta_rival = 0.35  ->  exp(0.35) ≈ 1.42  (수요 약 42% 상승)
    pair = (row['home_team'], row['away_team'])
    if pair in RIVALRY_PAIRS or pair[::-1] in RIVALRY_PAIRS:
        delta_rival = 0.35
        log_shock += delta_rival

    return log_shock

def run_evaluation_pipeline(df, M0_df, CAPACITY_df, indicator_fn = my_virtual_indicators,
                             shock_clip=SHOCK_CLIP, verbose_clip=True):
    d = df.copy()
    d = d.merge(M0_df.rename(columns={'team': 'home_team'}),
                on=['home_team', 'season'], how='left')
    d = d.merge(CAPACITY_df.rename(columns={'team': 'home_team'}),
                on=['home_team', 'season'], how='left')

    results = []
    for _, row in d.iterrows():
        raw_log_shock = float(indicator_fn(row))
        log_shock = float(np.clip(raw_log_shock, *shock_clip))
        is_clipped = abs(raw_log_shock - log_shock) > 1e-9
        M_t = row['M0'] * np.exp(log_shock)

        res = optimize_single_game(M_t, row['capacity'])
        det = revenue_detail(res.x, M_t, row['capacity'])

        _lo = CONFIG['p_min_ratio']; _hi = CONFIG['p_max_ratio']
        _bases = [BASE_PRICES['일반석'], BASE_PRICES['프리미엄석'],
                  BASE_PRICES['일반석'], BASE_PRICES['프리미엄석']]
        corner_hits = sum(
            1 for _p, _b in zip(res.x, _bases)
            if (_p / _b) >= _hi - 5e-3 or (_p / _b) <= _lo + 5e-3
        )

        results.append({
            'game_id':        f"{row['home_team']} vs {row['away_team']}",
            'M0':              row['M0'],
            'raw_log_shock':   round(raw_log_shock, 4),
            'log_shock':       round(log_shock, 4),
            'clipped':         is_clipped,
            'shock_x':         round(np.exp(log_shock), 3),
            'M_t':             round(M_t),
            'capacity':        row['capacity'],
            'p_D7_G':    round(res.x[0]), 'p_D7_P': round(res.x[1]),
            'p_D3_G':    round(res.x[2]), 'p_D3_P': round(res.x[3]),
            'audience':  round(det['total_audience']),
            'revenue':   round(det['revenue']),
            'corner_hits': corner_hits,
            'success':   res.success,
        })
    out = pd.DataFrame(results)

    if verbose_clip:
        clipped_rows = out[out['clipped']]
        if len(clipped_rows) > 0:
            lo, hi = shock_clip
            print(f"[SHOCK_CLIP 경고] {len(clipped_rows)}개 경기에서 log_shock이 "
                  f"허용 범위 ({lo}, {hi})를 벗어나 잘렸습니다:")
            for _, r in clipped_rows.iterrows():
                print(f"   - {r['game_id']:<24} 원래 합 δ = {r['raw_log_shock']:>7.4f}  "
                      f"->  적용된 값 = {r['log_shock']:>6.2f}")
            print("   지표 계수를 조정해 이 범위 안에서 의도한 효과가 나도록 다시 설계해 보세요.\n")

        corner_rows = out[out['corner_hits'] > 0]
        if len(corner_rows) > 0:
            print(f"[코너해 경고] {len(corner_rows)}개 경기에서 최적가가 가격 경계(0.7x/1.5x)에 "
                  f"붙었습니다 (corner_hits > 0):")
            for _, r in corner_rows.iterrows():
                print(f"   - {r['game_id']:<24} corner_hits = {int(r['corner_hits'])}  "
                      f"(log_shock = {r['log_shock']:>6.2f})")
            print("   이 행들은 '더 올리고/내리고 싶은데 경계에 막힌' 코너해라 진짜 내부 최적해가\n"
                  "   아닙니다. 대조군-처치군 비교가 왜곡되므로 지표(δ) 설계를 재검토하세요.\n")

    return out

In [55]:
# ==============================================================================
# 9. [실행] 시나리오 시뮬레이션
# ==============================================================================
res = run_evaluation_pipeline(df_games_demo, M0_df, CAPACITY_df)

bG, bP = BASE_PRICES['일반석'], BASE_PRICES['프리미엄석']
view = res.copy()
for c, b in [('p_D7_G', bG), ('p_D7_P', bP), ('p_D3_G', bG), ('p_D3_P', bP)]:
    view[c + '_x'] = (view[c] / b).round(2)

print('=' * 88)
print(' [버전 A] 라이벌전 vs 평상시  —  M_t = M_0 * exp(log_shock)')
print('=' * 88)
print(view[['game_id', 'M0', 'shock_x', 'M_t', 'capacity',
            'p_D7_G_x', 'p_D7_P_x', 'p_D3_G_x', 'p_D3_P_x',
            'audience', 'revenue', 'corner_hits']].to_string(index=False))

rev_rival  = res.loc[res['game_id'].str.contains('Rams'),    'revenue'].values[0]
rev_normal = res.loc[res['game_id'].str.contains('Falcons'), 'revenue'].values[0]

print()
print('-' * 88)
print(f"  평상시  (Jaguars vs Falcons) 기대수익 : {rev_normal:>15,} 원   (지수 100.0)")
print(f"  라이벌전 (Jaguars vs Rams)    기대수익 : {rev_rival:>15,} 원   "
      f"(지수 {rev_rival / rev_normal * 100:.1f})")
print(f"  수익 증가율 : {(rev_rival / rev_normal - 1) * 100:+.1f}%")
print('-' * 88)

 [버전 A] 라이벌전 vs 평상시  —  M_t = M_0 * exp(log_shock)
           game_id    M0  shock_x   M_t  capacity  p_D7_G_x  p_D7_P_x  p_D3_G_x  p_D3_P_x  audience   revenue  corner_hits
   Jaguars vs Rams 17575    1.419 24940     23750      1.25      1.20      1.39      1.27     15516 346332772            0
Jaguars vs Falcons 17575    1.000 17575     23750      1.33      1.11      1.41      1.18     10912 265107694            0

----------------------------------------------------------------------------------------
  평상시  (Jaguars vs Falcons) 기대수익 :     265,107,694 원   (지수 100.0)
  라이벌전 (Jaguars vs Rams)    기대수익 :     346,332,772 원   (지수 130.6)
  수익 증가율 : +30.6%
----------------------------------------------------------------------------------------


## [Example] 추가 분석 진행

In [56]:
# ==============================================================================
# 10. [진단] D-7 가격 역전 현상 해부
#      "수요가 늘었는데 왜 D-7 가격은 오히려 내려가는가?"
# ==============================================================================
print('=' * 88)
print(' [진단] 시점별 내부 상태 비교')
print('=' * 88)

for _, r in res.iterrows():
    px = [r['p_D7_G'], r['p_D7_P'], r['p_D3_G'], r['p_D3_P']]
    d = revenue_detail(px, r['M_t'], r['capacity'])

    print(f"\n▶ {r['game_id']}   (M_t = {r['M_t']:,} / capacity = {r['capacity']:,})")
    print(f"   시장배분   : M_D7 = {d['M_D7']:>9,.0f}   M_D3 = {d['M_D3']:>9,.0f}")
    print(f"   좌석용량   : cap_G = {d['cap_G']:>8,.0f}   cap_P = {d['cap_P']:>8,.0f}")
    print(f"   D-7 가격   : G {r['p_D7_G']:>7,} ({r['p_D7_G']/bG:.2f}x)   "
          f"P {r['p_D7_P']:>7,} ({r['p_D7_P']/bP:.2f}x)")
    print(f"   D-7 판매   : G {d['Q_D7_G']:>8,.0f}          P {d['Q_D7_P']:>8,.0f}")
    print(f"   ─ 잔여재고율 r_G = {d['r_G']:.3f}  →  희소성 f_G = {d['f_G']:.3f}  "
          f"→  유효민감도 γ_e,G = {d['gamma_eff_G_D3']:.6f}")
    print(f"   D-3 가격   : G {r['p_D3_G']:>7,} ({r['p_D3_G']/bG:.2f}x)   "
          f"P {r['p_D3_P']:>7,} ({r['p_D3_P']/bP:.2f}x)")
    print(f"   D-3 판매   : G {d['Q_D3_G']:>8,.0f}          P {d['Q_D3_P']:>8,.0f}")
    print(f"   총 관중    : {d['total_audience']:>8,.0f}      수익 : {d['revenue']:>15,.0f} 원")

print('\n' + '─' * 88)
print(' 해석: 라이벌전은 D-3 시장(M_D3)이 훨씬 크다. 따라서 최적해는 D-7 가격을')
print('       상대적으로 낮춰 재고를 미리 소진시키고(잔여재고율↓ → 희소성 f↑ →')
print('       유효민감도 γ_e↓), 그 결과 가격 저항이 약해진 D-3에서 더 많은 물량을')
print('       더 비싸게 파는 전략을 택한다. "수요가 늘면 무조건 올린다"는 직관과 다르다.')
print()
print(' [중요] 위 가격은 모두 0.7x~1.5x 경계에 붙지 않은 "내부해"이다.')
print('        만약 어떤 가격이 1.50x 에 딱 붙어 있다면, 그것은 최적해가 아니라')
print('        "더 올리고 싶은데 상한에 막힌" 코너해이며, 시나리오 간 비교가 왜곡된다.')
print('─' * 88)

 [진단] 시점별 내부 상태 비교

▶ Jaguars vs Rams   (M_t = 24,940 / capacity = 23,750)
   시장배분   : M_D7 =     9,976   M_D3 =    14,964
   좌석용량   : cap_G =   20,188   cap_P =    3,562
   D-7 가격   : G  15,022 (1.25x)   P  42,008 (1.20x)
   D-7 판매   : G    4,890          P    1,448
   ─ 잔여재고율 r_G = 0.758  →  희소성 f_G = 1.073  →  유효민감도 γ_e,G = 0.000168
   D-3 가격   : G  16,691 (1.39x)   P  44,530 (1.27x)
   D-3 판매   : G    7,063          P    2,114
   총 관중    :   15,516      수익 :     346,331,422 원

▶ Jaguars vs Falcons   (M_t = 17,575 / capacity = 23,750)
   시장배분   : M_D7 =     7,030   M_D3 =    10,545
   좌석용량   : cap_G =   20,188   cap_P =    3,562
   D-7 가격   : G  15,946 (1.33x)   P  38,859 (1.11x)
   D-7 판매   : G    2,963          P    1,465
   ─ 잔여재고율 r_G = 0.853  →  희소성 f_G = 1.044  →  유효민감도 γ_e,G = 0.000172
   D-3 가격   : G  16,954 (1.41x)   P  41,274 (1.18x)
   D-3 판매   : G    4,386          P    2,097
   총 관중    :   10,911      수익 :     265,106,784 원

─────────────────────────────────────────────

In [57]:
# ==============================================================================
# 11. [부록] 균일 가격배수 스윕 — 최적화 없이 단순 인상하면?
# ==============================================================================
row_r = df_games_demo.iloc[0]
M0_r  = M0_df.query("team=='Jaguars' and season==2024")['M0'].values[0]
cap_r = CAPACITY_BASE['Jaguars']
M_t_r = M0_r * np.exp(my_virtual_indicators(row_r))

print(f"라이벌전 (M_t = {M_t_r:,.0f}, capacity = {cap_r:,})")
print(f"{'배수':>6} | {'수익(원)':>16} | {'관중':>9} | {'지수':>7}")
print('-' * 50)

base_rev = revenue_detail([bG, bP, bG, bP], M_t_r, cap_r)['revenue']
for m in [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    d = revenue_detail([bG * m, bP * m, bG * m, bP * m], M_t_r, cap_r)
    print(f"{m:6.1f} | {d['revenue']:16,.0f} | {d['total_audience']:9,.0f} | "
          f"{100 * d['revenue'] / base_rev:7.1f}")

opt = optimize_single_game(M_t_r, cap_r)
d_opt = revenue_detail(opt.x, M_t_r, cap_r)
print('-' * 50)
print(f"{'최적화':>6} | {d_opt['revenue']:16,.0f} | {d_opt['total_audience']:9,.0f} | "
      f"{100 * d_opt['revenue'] / base_rev:7.1f}")
print('\n→ 균일 인상으로는 최적해에 도달할 수 없다. 시점·등급별 차등이 핵심이다.')

라이벌전 (M_t = 24,940, capacity = 23,750)
    배수 |            수익(원) |        관중 |      지수
--------------------------------------------------
   0.7 |      199,043,265 |    16,868 |    67.8
   0.8 |      231,797,576 |    17,317 |    79.0
   0.9 |      263,594,005 |    17,579 |    89.9
   1.0 |      293,366,810 |    17,619 |   100.0
   1.1 |      320,076,451 |    17,420 |   109.1
   1.2 |      341,298,418 |    16,944 |   116.3
   1.3 |      325,612,568 |    15,537 |   111.0
   1.4 |      305,857,329 |    14,061 |   104.3
   1.5 |      283,023,582 |    12,553 |    96.5
--------------------------------------------------
   최적화 |      346,332,772 |    15,516 |   118.1

→ 균일 인상으로는 최적해에 도달할 수 없다. 시점·등급별 차등이 핵심이다.


# [참가자 작성 영역] 가상지표 정의

### 수식 설명
$$\log(\text{shock}_t) = \sum_j \delta_j \, V_{j,t} \qquad\Longrightarrow\qquad M_t = M_0 \cdot e^{\log(\text{shock}_t)}$$

| 기호 | 의미 | 부호 해석 |
|---|---|---|
| $V_{j,t}$ | 가상지표 $j$의 발동 여부(0/1) 또는 강도 | — |
| $\delta_j$ | 지표 $j$의 **로그 스케일 충격 계수** | $\delta>0$ 수요↑ / $\delta<0$ 수요↓ |
| $e^{\delta}$ | 실제 **수요 배수** | $\delta=0.35 \Rightarrow e^{0.35}\approx1.42$ (약 42% 증가) |

In [58]:
# ==============================================================================
# Cell 12. [참가자 수정 영역]
# 최종 가상지표 모델: 구조지표 V1~V2 + 이벤트지표 V3~V5
#
# 고정모형
#     M_t = M_0 * exp(log_shock)
#
# 설계 원칙
# - 구조지표 V1, V2는 모든 경기에 적용한다.
# - 이벤트지표 V3, V4, V5는 해당 관측값이 있는 경기에만 적용한다.
# - 각 효과는 log_shock에 한 번만 더한다.
# - 수요배수를 M0 또는 Mt에 별도로 다시 곱하지 않는다.
# - SHOCK_CLIP과 가격 최적화는 이후 출제자 고정 셀에서 처리한다.
#
# 사후정보 제외
# - rain_at_game_time 사용 안 함
# - temp_at_game_time 사용 안 함
# ==============================================================================


def _is_one(value):
    """
    0/1 플래그가 1인지 결측치와 자료형 오류에 안전하게 판별한다.
    """
    try:
        return pd.notna(value) and float(value) == 1.0
    except (TypeError, ValueError):
        return False


def virtual_indicator_components(row, cfg=CONFIG):
    """
    V1~V5의 로그충격을 구조효과와 이벤트효과로 분리하여 계산한다.

    Returns
    -------
    dict
        structural_shock : V1 요일 + V2 시즌 시기
        event_shock      : V3 대기록 + V4 부상 + V5 강수확률
        log_shock        : structural_shock + event_shock
    """

    structural_shock = 0.0
    event_shock = 0.0

    # ==========================================================================
    # V1. 요일 효과
    #
    # 최신 제안서 설계값
    # 화요일 -0.15
    # 수요일 -0.12
    # 목요일 -0.12
    # 금요일 +0.03
    # 토요일 +0.20
    # 일요일 +0.10
    #
    # 월요일은 KBO 정규 편성 실측표본이 없으므로
    # 가장 가까운 평일인 화요일과 같은 -0.15를 적용한다.
    # ==========================================================================

    weekday_effect = {
        "월": -0.15,
        "화": -0.15,
        "수": -0.12,
        "목": -0.12,
        "금":  0.03,
        "토":  0.20,
        "일":  0.10,
    }

    day_text = str(
        row.get("day_of_week", "")
    ).strip()

    if day_text in weekday_effect:
        structural_shock += weekday_effect[day_text]

    else:
        # day_of_week 값이 없거나 예상 형식이 아닐 때만
        # date에서 실제 요일을 계산하여 보조적으로 사용한다.
        game_date = pd.to_datetime(
            row.get("date"),
            errors="coerce"
        )

        if pd.notna(game_date):
            weekday_num_effect = {
                0: -0.15,  # 월요일
                1: -0.15,  # 화요일
                2: -0.12,  # 수요일
                3: -0.12,  # 목요일
                4:  0.03,  # 금요일
                5:  0.20,  # 토요일
                6:  0.10,  # 일요일
            }

            structural_shock += weekday_num_effect.get(
                int(game_date.dayofweek),
                0.0
            )

    # ==========================================================================
    # V2. 시즌 시기 효과
    #
    # 3월: +0.05
    # 4월: -0.05
    # 5월 이후: 0
    #
    # 9월은 확보한 근거의 방향이 일치하지 않으므로 0으로 둔다.
    # ==========================================================================

    game_date = pd.to_datetime(
        row.get("date"),
        errors="coerce"
    )

    if pd.notna(game_date):
        season_effect = {
            3:  0.05,
            4: -0.05,
        }

        structural_shock += season_effect.get(
            int(game_date.month),
            0.0
        )

    # ==========================================================================
    # V3. 스타 홈런 대기록 임박
    #
    # is_star_record == 1인 경기만 적용
    #
    # d = 대기록 달성까지 남은 홈런 수
    #
    # V3(d) = 0.08 + 0.22 × 0.60^d
    #
    # d=3: 약 0.128
    # d=2: 약 0.159
    # d=1: 약 0.212
    # d=0: 0.300
    # ==========================================================================

    if _is_one(
        row.get("is_star_record", 0)
    ):
        d = row.get("hr_from_record")

        if pd.notna(d):
            d = max(
                0.0,
                float(d)
            )

            star_log_shock = (
                0.08
                + 0.22 * (0.60 ** d)
            )

            event_shock += star_log_shock

    # ==========================================================================
    # V4. 핵심타자 부상 이탈
    #
    # is_key_injury == 1인 경기만 적용
    #
    # days = 부상 발생 후 경과일
    #
    # V4(days) = -0.12 × 0.50^(days / 30)
    #
    # 부상 직후 최대 충격은 -0.12이며
    # 30일마다 충격이 절반으로 감소한다.
    # ==========================================================================

    if _is_one(
        row.get("is_key_injury", 0)
    ):
        days = row.get("days_since_injury")

        if pd.notna(days):
            days = max(
                0.0,
                float(days)
            )

            injury_log_shock = (
                -0.12
                * (0.50 ** (days / 30.0))
            )

            event_shock += injury_log_shock

    # ==========================================================================
    # V5. 경기별 강수확률 예보
    #
    # rain_prob이 결측이 아닌 모든 경기에 적용
    #
    # r = 경기별 강수확률, 0 <= r <= 1
    # 실제 우천 시 평균 수요손실률 = 0.10
    #
    # 비우천 수요배수 = 1.00
    # 우천 수요배수   = 0.90
    #
    # 기대수요배수
    # = (1-r) × 1.00 + r × 0.90
    # = 1 - 0.10r
    #
    # 고정모형이 M_t = M_0 × exp(log_shock)이므로
    #
    # V5(r) = ln(1 - 0.10r)
    #
    # 주의
    # - rain_prob_high는 발동조건으로 사용하지 않는다.
    # - 5% 기준 정규화를 하지 않는다.
    # - 대조군 r=0.05에도 ln(0.995)가 적용된다.
    # - 기대수요배수를 M0 또는 Mt에 다시 곱하지 않는다.
    # - D-7/D-3 판매비율을 날씨 가중치로 사용하지 않는다.
    # - 별도 임계값이나 추가 시간감쇠를 적용하지 않는다.
    # ==========================================================================

    rain_prob = row.get("rain_prob")

    if pd.notna(rain_prob):
        r = float(
            np.clip(
                float(rain_prob),
                0.0,
                1.0
            )
        )

        RAIN_LOSS = 0.10

        rain_log_shock = np.log1p(
            -RAIN_LOSS * r
        )

        event_shock += rain_log_shock

    # ==========================================================================
    # 최종 로그충격
    # ==========================================================================

    total_log_shock = (
        structural_shock
        + event_shock
    )

    return {
        "structural_shock": float(structural_shock),
        "event_shock": float(event_shock),
        "log_shock": float(total_log_shock),
    }


# ==============================================================================
# A. 초기 베이스라인
# 가상지표를 전혀 적용하지 않는다.
# ==============================================================================

def zero_indicator(row, cfg=CONFIG):
    return 0.0


# ==============================================================================
# B. 구조지표 전용
# V1 요일 + V2 시즌 시기만 적용한다.
# ==============================================================================

def structural_only_indicator(row, cfg=CONFIG):
    components = virtual_indicator_components(
        row,
        cfg
    )

    return components["structural_shock"]


# 기존 명칭을 사용하는 분석 코드가 있을 경우를 위한 호환 함수
def common_only_indicator(row, cfg=CONFIG):
    return structural_only_indicator(
        row,
        cfg
    )


# ==============================================================================
# C. 이벤트지표 전용
# V3 대기록 + V4 부상 + V5 강수확률만 적용한다.
# ==============================================================================

def event_only_indicator(row, cfg=CONFIG):
    components = virtual_indicator_components(
        row,
        cfg
    )

    return components["event_shock"]


# ==============================================================================
# D. 최종 제출 모델
# V1 + V2 + V3 + V4 + V5 전체 적용
# ==============================================================================

def my_virtual_indicators(row, cfg=CONFIG):
    components = virtual_indicator_components(
        row,
        cfg
    )

    return components["log_shock"]

In [59]:
# ==============================================================================
# 13. [출제자 고정 영역] Phase 3 실제 제시 시나리오 데이터셋 (참가자 배포용)
#
#  시나리오별 규모
#   1. 긍정 (스타 대기록)    : 대조 4경기 + 처치 4경기 (7일 홈 4연전)
#   2. 부정I (핵심타자 부상)  : 대조 5경기 + 처치 5경기 (2개월 구간 대표 샘플)
#   3. 부정II (기습장마 예보) : 대조 6경기 + 처치 6경기 (1~2주 구간 대표 샘플)
#
#  컬럼 설명
#   - scenario: 3개 시나리오 구분
#   - pair: 대조군(비교 기준 경기) / 처치군(이벤트가 실제로 걸린 경기)
#   - date, season: 경기 날짜, 시즌
#   - home_team, away_team: 홈/원정팀
#   - day_of_week: 요일
#   - home_rank, away_rank: 홈/원정팀 순위
#   - is_star_record / is_key_injury / rain_prob_high : 해당 구간에 속하는지 여부 플래그
#     (사실관계 — 조직위 고정). 구간 내 여러 경기에 동일하게 1로 표시된다.
#   - hr_from_record    : [시나리오1] 대기록까지 남은 홈런 수 (작을수록 임박, 0=달성).
#                          ※ 이 값은 경기 결과를 사후 반영한 수치가 아니라, 각 경기가
#                          시작되기 전 이미 언론·구단 발표 등을 통해 공표되어 있던
#                          잔여 홈런 수(pregame 공표치)라는 조직위 설정(시나리오 가정)이다.
#                          즉 D-7/D-3 가격 결정 시점에 참가자가 알 수 있는 사전 정보로
#                          간주하고 사용한다.
#   - days_since_injury  : [시나리오2] 부상 발생 후 경과일 (작을수록 최근). 부상 발생일
#                          이라는 고정 사건 기준 달력상 계산값이다.
#   - rain_prob          : [시나리오3] 해당 경기의 예보 강수확률 (연속값 0~1). 가격 결정
#                          시점 기준 최신 기상예보치이다.
#
#  ※ temp_at_game_time(경기 시각 기온), rain_at_game_time(경기 당일 실제 강수 여부)은
#    모두 경기가 실제로 열려야 확정되는 당일 실측치로, D-7/D-3 가격 결정 시점에는
#    관측이 불가능한 사후 정보다. 시나리오 관련성과 무관하게 전 시나리오 공통으로
#    참가자 배포용 데이터셋에서 컬럼 자체를 삭제한다.
#
#  참가자는 flag(0/1)만 보고 구간 전체에 동일한 δ를 적용(일괄적용)할 수도 있고,
#  hr_from_record / days_since_injury / rain_prob 같은 연속값으로 경기별 강도를 달리
#  반영(선별적용)할 수도 있다. 어느 쪽을 택하든 그 근거를 서술하는 것이 채점 대상이며,
#  baseline은 이 3개 시나리오를 풀어주지 않는다.
# ==============================================================================
df_games_scenarios = pd.DataFrame([
    # ==== 시나리오 1: 긍정 (스타 선수 홈런 대기록 임박) — 대조 4 + 처치 4 ====
    # 대조군: 시즌 초반, 대기록 이슈 없는 평범한 4연전
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '대조군', 'date': '2024-04-05', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '금',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '대조군', 'date': '2024-04-06', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '대조군', 'date': '2024-04-07', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '대조군', 'date': '2024-04-08', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    # 처치군: 7일 뒤 홈 4연전, 대기록까지 3->0개로 카운트다운 (마지막 경기에서 달성)
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '처치군', 'date': '2024-06-14', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '금',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 1, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': 3, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '처치군', 'date': '2024-06-15', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 1, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': 2, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '처치군', 'date': '2024-06-16', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 1, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': 1, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '처치군', 'date': '2024-06-17', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 1, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': 0, 'days_since_injury': None, 'rain_prob': None},

    # ==== 시나리오 2: 부정I (핵심 타자 부상 이탈, 2개월) — 대조 5 + 처치 5 ====
    # 대조군: 부상 이슈 없는 시즌 초반 대표 샘플 5경기
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-03-23', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-03-24', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-03-30', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-03-31', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-04-13', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    # 처치군: 부상(2024-07-01) 이후 2개월 구간, 경과일 2/14/30/45/58일 대표 샘플
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-07-03', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '수',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 2, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-07-15', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 14, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-07-31', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '수',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 30, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-08-15', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '목',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 45, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-08-28', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '수',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 58, 'rain_prob': None},

    # ==== 시나리오 3: 부정II (기습 장마 예보, 1~2주) — 대조 6 + 처치 6 ====
    # 대조군: 맑은 날씨가 이어지는 시즌 중반 대표 샘플 6경기
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-19', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '금',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-20', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-21', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-26', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '금',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-27', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-28', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    # 처치군: 9/1 기습 예보 이후 1~2주 구간, 강수확률이 0.85 -> 0.20으로 점차 낮아짐
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-02', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.85},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-03', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '화',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.75},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-04', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '수',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.65},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-08', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.50},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-09', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.35},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-10', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '화',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.20}
])
df_games_scenarios

,scenario,pair,date,season,home_team,away_team,day_of_week,home_rank,away_rank,is_star_record,is_key_injury,rain_prob_high,hr_from_record,days_since_injury,rain_prob
0,1. 긍정 (스타 대기록),대조군,2024-04-05,2024,Jaguars,Falcons,금,2,5,0,0,0,NaN,NaN,NaN
1,1. 긍정 (스타 대기록),대조군,2024-04-06,2024,Jaguars,Falcons,토,2,5,0,0,0,NaN,NaN,NaN
2,1. 긍정 (스타 대기록),대조군,2024-04-07,2024,Jaguars,Falcons,일,2,5,0,0,0,NaN,NaN,NaN
3,1. 긍정 (스타 대기록),대조군,2024-04-08,2024,Jaguars,Falcons,월,2,5,0,0,0,NaN,NaN,NaN
4,1. 긍정 (스타 대기록),처치군,2024-06-14,2024,Jaguars,Falcons,금,2,5,1,0,0,3.0,NaN,NaN
5,1. 긍정 (스타 대기록),처치군,2024-06-15,2024,Jaguars,Falcons,토,2,5,1,0,0,2.0,NaN,NaN
6,1. 긍정 (스타 대기록),처치군,2024-06-16,2024,Jaguars,Falcons,일,2,5,1,0,0,1.0,NaN,NaN
7,1. 긍정 (스타 대기록),처치군,2024-06-17,2024,Jaguars,Falcons,월,2,5,1,0,0,0.0,NaN,NaN
8,2. 부정I (핵심타자 부상),대조군,2024-03-23,2024,Jaguars,Falcons,토,2,5,0,0,0,NaN,NaN,NaN
9,2. 부정I (핵심타자 부상),대조군,2024-03-24,2024,Jaguars,Falcons,일,2,5,0,0,0,NaN,NaN,NaN


---
## [실행] df_games_scenarios 파이프라인 실행

`df_games_demo`용 cell 10~12(게임명 문자열로 'Rams'/'Falcons'를 찾는 방식)는 `df_games_scenarios`
구조(여러 시나리오 × 여러 경기)에 맞지 않는다. 대신 `scenario`/`pair` 컬럼으로 그룹핑해서 보여주는
별도 실행 함수를 아래에 둔다.

본인이 작성한 `my_virtual_indicators()`를 다 채운 뒤, 아래 셀을 그대로 실행하면 시나리오별
경기 단위 결과와 대조군-처치군 비교 요약이 출력된다. 지금은 라이벌전 지표만 구현돼 있어
3개 시나리오 전부 동일한 결과(log_shock=0)가 나오는 게 정상이다 — 본인의 지표를 추가하면서
이 셀을 다시 실행해 값이 어떻게 바뀌는지 직접 확인하는 용도로 쓰면 된다.

In [60]:
# ==============================================================================
# 14. [실행] Phase 3 시나리오 데이터셋(df_games_scenarios) 파이프라인 실행
# ==============================================================================
def run_scenario_pipeline(indicator_fn=my_virtual_indicators):
    """df_games_scenarios를 run_evaluation_pipeline에 태우고,
    scenario/pair 컬럼을 붙여 시나리오별로 읽기 좋게 정리해서 돌려준다."""
    res = run_evaluation_pipeline(df_games_scenarios, M0_df, CAPACITY_df, indicator_fn=indicator_fn)
    res = res.reset_index(drop=True)
    meta = df_games_scenarios[['scenario', 'pair', 'date']].reset_index(drop=True)
    out = pd.concat([meta, res.drop(columns=['capacity'])], axis=1)  # M0는 유지, capacity만 제거
    return out


def print_scenario_results(indicator_fn=my_virtual_indicators, label='my_virtual_indicators'):
    """cell 10([실행] 시나리오 시뮬레이션)과 동일한 포맷으로 df_games_scenarios 결과를 출력한다."""
    res = run_scenario_pipeline(indicator_fn)

    bG, bP = BASE_PRICES['일반석'], BASE_PRICES['프리미엄석']
    view = res.copy()
    for c, b in [('p_D7_G', bG), ('p_D7_P', bP), ('p_D3_G', bG), ('p_D3_P', bP)]:
        view[c + '_x'] = (view[c] / b).round(2)

    print('=' * 100)
    print(f' [{label}] df_games_scenarios 시뮬레이션  —  M_t = M_0 * exp(log_shock)')
    print('=' * 100)
    print(view[['scenario', 'pair', 'date', 'M0', 'raw_log_shock', 'log_shock', 'shock_x', 'M_t',
                'p_D7_G_x', 'p_D7_P_x', 'p_D3_G_x', 'p_D3_P_x',
                'audience', 'revenue', 'corner_hits']].to_string(index=False))
    n_clipped = int(view['clipped'].sum())
    if n_clipped > 0:
        print(f"\n※ 위 표에서 raw_log_shock(설계한 원래 값)과 log_shock(SHOCK_CLIP 적용 후 값)이"
              f" 다른 행이 {n_clipped}개 있습니다 — 해당 행은 clip되어 delta 총합이 잘렸습니다.")

    print()
    print('-' * 100)
    print(' [시나리오별 대조군 vs 처치군 비교 (구간 평균 기준)]')
    print('-' * 100)
    summary = res.groupby(['scenario', 'pair'])['revenue'].mean().unstack()
    for scenario in summary.index:
        rev_control = summary.loc[scenario, '대조군']
        rev_treat = summary.loc[scenario, '처치군']
        idx = rev_treat / rev_control * 100
        pct = (rev_treat / rev_control - 1) * 100
        print(f"  {scenario:<22} 대조군 {rev_control:>15,.0f}원 (지수 100.0)  ->  "
              f"처치군 {rev_treat:>15,.0f}원 (지수 {idx:6.1f}, {pct:+.1f}%)")
    print('-' * 100)

    return res


res_scenarios = print_scenario_results(my_virtual_indicators, 'my_virtual_indicators')

 [my_virtual_indicators] df_games_scenarios 시뮬레이션  —  M_t = M_0 * exp(log_shock)
         scenario pair       date    M0  raw_log_shock  log_shock  shock_x   M_t  p_D7_G_x  p_D7_P_x  p_D3_G_x  p_D3_P_x  audience   revenue  corner_hits
   1. 긍정 (스타 대기록)  대조군 2024-04-05 17575        -0.0200    -0.0200    0.980 17227      1.33      1.11      1.42      1.17     10697 261179266            0
   1. 긍정 (스타 대기록)  대조군 2024-04-06 17575         0.1500     0.1500    1.162 20419      1.29      1.15      1.40      1.22     12681 296825449            0
   1. 긍정 (스타 대기록)  대조군 2024-04-07 17575         0.0500     0.0500    1.051 18476      1.32      1.12      1.41      1.19     11470 275225527            0
   1. 긍정 (스타 대기록)  대조군 2024-04-08 17575        -0.2000    -0.2000    0.819 14389      1.38      1.06      1.44      1.12      8948 228580738            0
   1. 긍정 (스타 대기록)  처치군 2024-06-14 17575         0.1575     0.1575    1.171 20573      1.29      1.15      1.40      1.22     12778 298528160         